# Biotech Catalyst EDA
### Distributions of ATR, Stock Movement, and ATR-Normalized Move

This notebook explores three core signals:
1. **`atr_pct`** — A stock's *typical* daily price range as % of price (its normal volatility baseline)
2. **`move_pct`** — The actual % move on the catalyst event day
3. **`normalized_move`** — How many ATRs the event move represents (`move_pct / atr_pct`) — the cleanest measure of *how surprising* the move was relative to the stock's own history

The dataset covers **2,175 clinical catalyst events** across 341 biotech tickers (2007–2026).

---

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

TEMPLATE = 'plotly_white'
COLORS = {
    'Gainer': '#2ecc71',
    'Loser': '#e74c3c',
    'Low': '#74b9ff',
    'Medium': '#fdcb6e',
    'High': '#d63031',
    'Noise': '#b2bec3',
    'Extreme': '#6c5ce7',
}

# Load from parent directory
import os
os.chdir(os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())

df_raw = pd.read_csv('enriched_all_clinical.csv')

# Clean working copy: ML-ready rows
df = df_raw[df_raw['data_complete'] == True].copy()
df['abs_move_pct'] = df['move_pct'].abs()
df['log_market_cap'] = np.log10(df['market_cap_m'].clip(lower=1))
df['market_cap_bucket'] = pd.cut(
    df['market_cap_m'],
    bins=[0, 100, 500, 2000, np.inf],
    labels=['Micro (<$100M)', 'Small ($100–500M)', 'Mid ($500M–2B)', 'Large (>$2B)']
)
df['has_nct'] = df['nct_id'].notna() & df['nct_id'].astype(str).str.startswith('NCT')
df['phase_clean'] = df['ct_phase'].str.upper().str.replace('PHASE', 'Phase ', regex=False).str.strip()
df['phase_clean'] = df['phase_clean'].replace({'Phase  1': 'Phase 1', 'Phase  2': 'Phase 2',
                                                'Phase  3': 'Phase 3', 'Phase  4': 'Phase 4',
                                                'NA': 'Unknown', 'N/A': 'Unknown'})

print(f'Total events:         {len(df_raw):,}')
print(f'ML-ready (complete):  {len(df):,}  ({100*len(df)/len(df_raw):.1f}%)')
print(f'Gainers / Losers:     {(df["event_type"]=="Gainer").sum()} / {(df["event_type"]=="Loser").sum()}')
print(f'Date range:           {df_raw["event_date"].min()} → {df_raw["event_date"].max()}')

## 1 · Column Fill Rates
Before exploring distributions, let's confirm data coverage for the key columns.

In [ ]:
focus_cols = [
    ('atr_pct',          'ATR % (typical daily range)'),
    ('avg_daily_move',   'Avg daily close-to-close move %'),
    ('move_pct',         'Event-day stock move %'),
    ('normalized_move',  'Normalized move (X × ATR)'),
    ('move_class_norm',  'ATR-class label (Noise/Low/…/Extreme)'),
    ('stock_relative_move', 'ML label (Low / Medium / High)'),
    ('move_2d_pct',      '2-day move after event'),
    ('market_cap_m',     'Market cap $M'),
    ('short_percent',    'Short interest %'),
    ('ct_phase',         'Clinical trial phase'),
]

rows = []
for col, label in focus_cols:
    if col in df_raw.columns:
        n = df_raw[col].notna().sum()
        rows.append({'Column': col, 'Description': label,
                     'Filled': n, 'Total': len(df_raw),
                     'Fill %': round(100 * n / len(df_raw), 1)})

fill_df = pd.DataFrame(rows)

fig = px.bar(
    fill_df, x='Fill %', y='Column', orientation='h',
    color='Fill %', color_continuous_scale='Teal',
    text='Fill %', hover_data=['Description', 'Filled', 'Total'],
    title='Data Fill Rates — Key Columns',
    template=TEMPLATE, height=400,
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(coloraxis_showscale=False, xaxis_range=[0, 110],
                  yaxis={'categoryorder': 'total ascending'})
fig.show()
display(fill_df[['Column', 'Description', 'Filled', 'Fill %']])

## 2 · ATR Distribution
**`atr_pct`** is the stock's *baseline* daily volatility — how much it typically moves in a normal day.  
A 5% ATR means the stock moves ~5% on an average day before any catalyst.  
Small-cap biotech naturally has higher ATR than large pharma.

> **Why this matters**: A 20% move on a stock with 2% ATR is 10× its normal range — extraordinary.  
> The same 20% on a stock with 15% ATR is barely 1.3× — almost routine.

In [ ]:
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['ATR % Distribution (all events)', 'ATR % by Market Cap Tier'])

# Left: overall histogram
fig.add_trace(go.Histogram(
    x=df['atr_pct'].clip(upper=30), nbinsx=60,
    marker_color='#0984e3', opacity=0.8, name='ATR %',
    hovertemplate='ATR: %{x:.1f}%<br>Count: %{y}'
), row=1, col=1)

# Right: box by market cap bucket
order = ['Micro (<$100M)', 'Small ($100–500M)', 'Mid ($500M–2B)', 'Large (>$2B)']
cap_colors = ['#d63031', '#e17055', '#fdcb6e', '#00b894']
for i, (bucket, color) in enumerate(zip(order, cap_colors)):
    sub = df[df['market_cap_bucket'] == bucket]['atr_pct'].clip(upper=40)
    fig.add_trace(go.Box(
        y=sub, name=bucket, marker_color=color,
        boxpoints='outliers', jitter=0.3
    ), row=1, col=2)

# Stats annotation
atr = df['atr_pct'].dropna()
fig.add_vline(x=atr.median(), line_dash='dash', line_color='orange',
              annotation_text=f'Median {atr.median():.1f}%', row=1, col=1)
fig.add_vline(x=atr.mean(), line_dash='dot', line_color='red',
              annotation_text=f'Mean {atr.mean():.1f}%', row=1, col=1)

fig.update_layout(template=TEMPLATE, height=450, showlegend=False,
                  title_text='ATR % — Baseline Volatility Profile')
fig.update_xaxes(title_text='ATR %', row=1, col=1)
fig.update_yaxes(title_text='ATR %', row=1, col=2)
fig.show()

print(f'ATR % Summary:')
print(df['atr_pct'].describe().round(2).to_string())

## 3 · Stock Move Distribution (Event Day)
**`move_pct`** is the raw stock price change on the catalyst event day.  
Split by Gainers vs Losers to see the full picture — the distribution is strongly bimodal.

In [ ]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=['Full move_pct distribution', 'Absolute move | Gainer vs Loser']
)

# Left: full distribution with Gainer/Loser colors
for etype in ['Gainer', 'Loser']:
    sub = df[df['event_type'] == etype]['move_pct'].clip(-150, 200)
    fig.add_trace(go.Histogram(
        x=sub, name=etype, nbinsx=80, opacity=0.7,
        marker_color=COLORS[etype],
        hovertemplate=f'{etype}: %{{x:.1f}}%<br>Count: %{{y}}'
    ), row=1, col=1)

# Right: absolute move box by Gainer/Loser
for etype in ['Gainer', 'Loser']:
    sub = df[df['event_type'] == etype]['abs_move_pct'].clip(upper=150)
    fig.add_trace(go.Box(
        y=sub, name=etype, marker_color=COLORS[etype],
        boxpoints='outliers', jitter=0.2
    ), row=1, col=2)

fig.update_layout(template=TEMPLATE, height=450, barmode='overlay',
                  title_text='Event-Day Stock Move Distribution',
                  legend=dict(orientation='h', y=1.1))
fig.update_xaxes(title_text='move_pct (%)', row=1, col=1)
fig.update_yaxes(title_text='|move_pct| (%)', row=1, col=2)
fig.show()

for etype in ['Gainer', 'Loser']:
    sub = df[df['event_type'] == etype]['abs_move_pct']
    print(f'{etype:6s} — n={len(sub):4d} | median={sub.median():.1f}% | '
          f'mean={sub.mean():.1f}% | p90={sub.quantile(0.9):.1f}% | max={sub.max():.1f}%')

## 4 · Normalized Move: X × ATR
**`normalized_move` = `abs(move_pct)` / `atr_pct`** — the number of ATRs the event move represents.

This is the most important metric for comparing moves across stocks with different volatility profiles:
- `< 1.5×` = **Noise** — within normal daily range, no real signal
- `1.5 – 3×` = **Low** — slightly elevated, minor catalyst
- `3 – 5×` = **Medium** — clear catalyst signal
- `5 – 8×` = **High** — strong catalyst
- `≥ 8×` = **Extreme** — defining event (Phase 3 topline, FDA, etc.)

> Statistical note: ATR ≈ 0.8σ of daily returns, so 3× ATR ≈ 3.75σ — occurs by chance in <0.02% of normal trading days.

In [ ]:
nm = df['normalized_move'].clip(upper=30)

# Color each bar by which ATR class it falls in
bins = [0, 1.5, 3, 5, 8, 30]
labels_atr = ['Noise (<1.5×)', 'Low (1.5–3×)', 'Medium (3–5×)', 'High (5–8×)', 'Extreme (≥8×)']
palette = ['#b2bec3', '#74b9ff', '#fdcb6e', '#e17055', '#6c5ce7']
df['norm_bucket'] = pd.cut(df['normalized_move'], bins=bins, labels=labels_atr)

# specs required: col 1 = xy (histogram), col 2 = domain (pie)
fig = make_subplots(
    rows=1, cols=2,
    specs=[[{'type': 'xy'}, {'type': 'domain'}]],
    subplot_titles=['Normalized Move Distribution (X × ATR)', 'Distribution by ATR class']
)

# Left: histogram colored by ATR class bucket
for label, color in zip(labels_atr, palette):
    sub = df[df['norm_bucket'] == label]['normalized_move'].clip(upper=30)
    fig.add_trace(go.Histogram(
        x=sub, name=label, nbinsx=60,
        marker_color=color, opacity=0.85,
        hovertemplate=f'{label}: %{{x:.2f}}× ATR<br>Count: %{{y}}'
    ), row=1, col=1)

# Right: pie of ATR class counts
class_counts = df['move_class_norm'].value_counts()
class_order = ['Noise', 'Low', 'Medium', 'High', 'Extreme']
class_colors_pie = ['#b2bec3', '#74b9ff', '#fdcb6e', '#e17055', '#6c5ce7']
fig.add_trace(go.Pie(
    labels=[c for c in class_order if c in class_counts.index],
    values=[class_counts.get(c, 0) for c in class_order if c in class_counts.index],
    marker_colors=class_colors_pie,
    hole=0.4,
    textinfo='label+percent'
), row=1, col=2)

fig.update_layout(template=TEMPLATE, height=450, barmode='stack',
                  title_text='Normalized Move (move_pct ÷ atr_pct)',
                  legend=dict(orientation='h', y=-0.15))
fig.update_xaxes(title_text='Normalized move (× ATR)', row=1, col=1)
fig.show()

print('\nNormalized Move — percentiles:')
for p in [25, 50, 75, 90, 95, 99]:
    print(f'  p{p:2d}: {df["normalized_move"].quantile(p/100):.2f}× ATR')

## 5 · ATR vs Event Move — 2D Scatter

**What you're looking at:**
- **X axis** — `atr_pct`: the stock's *baseline* daily volatility (its normal daily range before the event)
- **Y axis** — `abs(move_pct)`: the actual % the stock moved on the catalyst event day (absolute value, so Gainers and Losers both appear positive)
- **Color** — `stock_relative_move`: the ML classification label (Low / Medium / High)

---

**Where does `stock_relative_move` come from?**

It is computed in `utils/volatility.py → classify_move()` and uses a **dual AND requirement** — the move must be large in *both* absolute terms and relative to the stock's own history:

| Label | Rule | Meaning |
|-------|------|---------|
| 🔵 **Low** | `abs(move) < 15%` **AND** `normalized_move < 2.5×` ATR | Small absolute move, and small relative to this stock's normal range |
| 🟡 **Medium** | Everything else | Either big in absolute terms but normal for this stock, OR small % but unusually large for a low-volatility name |
| 🔴 **High** | `abs(move) ≥ 20%` **AND** `normalized_move ≥ 3.5×` ATR | Meaningfully large move **AND** statistically unusual for this stock |

This is the same column as `move_class_combo` (aliased to `stock_relative_move` for clarity). It is the primary ML label used for training.

**The three diagonal reference lines** show where `move / ATR = 1×, 3×, 8×` — everything above the 3× line is a notable event, above 8× is extreme. Note how **High** events cluster above the 3× line, while **Low** events stay below 1×.

In [ ]:
plot_df = df[df['atr_pct'].notna() & df['abs_move_pct'].notna()].copy()
plot_df = plot_df[plot_df['atr_pct'] <= 35]
plot_df['stock_relative_move'] = pd.Categorical(
    plot_df['stock_relative_move'], categories=['Low', 'Medium', 'High'], ordered=True
)

fig = px.scatter(
    plot_df.sort_values('stock_relative_move'),
    x='atr_pct', y='abs_move_pct',
    color='stock_relative_move',
    color_discrete_map=COLORS,
    hover_data=['ticker', 'event_date', 'normalized_move', 'ct_phase', 'market_cap_m'],
    labels={
        'atr_pct':            'ATR % — baseline daily volatility (X axis)',
        'abs_move_pct':       'Stock % move on event day |absolute| (Y axis)',
        'stock_relative_move': 'ML label (Low / Medium / High)',
    },
    title='Event-Day Stock % Move vs ATR Baseline — colored by ML class',
    template=TEMPLATE, height=520, opacity=0.55
)

# Reference lines: y = N × x  →  where normalized_move = N
x_range = np.linspace(0, 35, 200)
for mult, dash, label, color in [
    (1,  'dot',  '1× ATR (normal daily noise)',  'rgba(100,100,100,0.4)'),
    (3,  'dash', '3× ATR (notable catalyst)',     'rgba(100,100,100,0.6)'),
    (8,  'solid','8× ATR (extreme event)',         'rgba(100,100,100,0.9)'),
]:
    fig.add_trace(go.Scatter(
        x=x_range, y=x_range * mult,
        mode='lines',
        line=dict(dash=dash, color=color, width=1.5),
        name=label, opacity=0.8
    ))

fig.update_layout(
    xaxis_range=[0, 35], yaxis_range=[0, 160],
    xaxis_title='ATR % (stock baseline volatility — how much it moves on a normal day)',
    yaxis_title='Stock % move on catalyst event day (absolute value)',
    legend=dict(orientation='h', y=1.05),
    annotations=[dict(
        x=32, y=155, text='Y = stock % move on event day (|move_pct|)',
        showarrow=False, font=dict(size=11, color='gray')
    )]
)
fig.show()

## 6 · Statistical Tests — What Drives Larger Normalized Moves?

We test which variables are most associated with `normalized_move` — the number of ATRs the event move represents.

### Why non-parametric tests?

`normalized_move` is **extremely right-skewed**: median ≈ 0.6×, mean ≈ 2×, max ≈ 39×.  
Standard parametric tests (ANOVA, Pearson correlation) assume a normal distribution — unreliable here.  
Non-parametric tests work on **ranks** of values rather than the values themselves, so they make no distributional assumptions.

---

### Test 1 — Kruskal-Wallis H &nbsp;*(categorical variables, ≥ 2 groups)*

**How it works**: Replace every `normalized_move` value with its rank in the full dataset (1 = smallest). Then measure whether the groups (e.g. Phase 1 vs Phase 2 vs Phase 3) have meaningfully different average ranks. If the groups were all identical, ranks would be evenly distributed; if one group consistently ranks higher, H grows large.

| Metric | Meaning |
|--------|---------|
| **H statistic** | Larger = groups are more separated in rank. H ≈ 0: all groups come from the same distribution. H = 100 means strong rank-separation. |
| **p-value** | Probability of observing this H by pure chance if the groups were identical. p < 0.05 = the separation is unlikely to be random. |
| **Limitation** | Tells you *that* groups differ, not *which specific pair* differs — follow up with pairwise comparisons if needed. |

---

### Test 2 — Spearman ρ &nbsp;*(continuous variables)*

**How it works**: Replace both X and Y with their rank orders, then compute the standard Pearson correlation on those ranks. Unlike Pearson, Spearman captures any **monotonic** relationship — does Y tend to increase as X increases, regardless of whether that relationship is linear?

| ρ value | Interpretation |
|---------|---------------|
| **+1** | Perfect positive rank correlation (higher X always → higher Y) |
| **0** | No monotonic relationship |
| **−1** | Perfect negative rank correlation |
| **±0.1** | Weak but often significant at n > 1,000 |
| **±0.3** | Moderate — practically meaningful |

---

### Why `stock_relative_move` is excluded

`stock_relative_move` is **derived directly from `normalized_move`** — it is a categorical bucket of the same number (`Low / Medium / High` based on thresholds applied to `normalized_move` and `abs(move_pct)`). Testing it against `normalized_move` would be circular, like testing whether °C predicts °F.

In [ ]:
from scipy.stats import kruskal, spearmanr

target = 'normalized_move'
kw_results = []
sp_results = []

# --- Kruskal-Wallis (categorical) ---
# stock_relative_move excluded: it is derived directly from normalized_move (circular)
cat_vars = [
    ('event_type',         'Gainer vs Loser'),
    ('phase_clean',        'Trial phase'),
    ('has_nct',            'Has NCT ID'),
    ('market_cap_bucket',  'Market cap tier'),
]
for col, label in cat_vars:
    if col not in df.columns:
        continue
    grouped = {str(k): g[target].dropna().values
               for k, g in df.groupby(col) if len(g) >= 10}
    if len(grouped) >= 2:
        stat, pval = kruskal(*grouped.values())
        kw_results.append({
            'Variable': label,
            'H statistic': round(stat, 1),
            'p-value': pval,
            'p_fmt': f'{pval:.1e}',
            'Significant': pval < 0.05,
            'grouped': grouped,
        })

# --- Spearman rho (continuous) ---
cont_vars = [
    ('market_cap_m',   'Market cap $M'),
    ('short_percent',  'Short interest %'),
    ('atr_pct',        'ATR % (own volatility)'),
    ('log_market_cap', 'Log10(market cap)'),
]
for col, label in cont_vars:
    if col not in df.columns:
        continue
    sub = df[[col, target]].dropna()
    if len(sub) >= 50:
        r, pval = spearmanr(sub[col], sub[target])
        sp_results.append({
            'Variable': label,
            'Spearman rho': round(r, 3),
            'p-value': pval,
            'p_fmt': f'{pval:.1e}',
            'n': len(sub),
            'Significant': pval < 0.05,
        })

kw_df_plot = pd.DataFrame(kw_results).sort_values('H statistic', ascending=True)
sp_df_plot = pd.DataFrame(sp_results).sort_values('Spearman rho')

# ── Chart 1: Kruskal-Wallis H ──────────────────────────────────────────────
fig_kw = px.bar(
    kw_df_plot, y='Variable', x='H statistic',
    text='p_fmt',
    color='H statistic', color_continuous_scale='Blues_r',
    orientation='h',
    title='Kruskal-Wallis H — Categorical Variables vs Normalized Move<br>'
          '<sup>Higher H = groups are more separated in their normalized_move rank-distribution. p-value on each bar.</sup>',
    labels={'H statistic': 'H statistic (higher = stronger group separation)'},
    template=TEMPLATE, height=320,
)
fig_kw.update_traces(textposition='outside', textfont_size=11)
fig_kw.update_layout(coloraxis_showscale=False,
                     xaxis_range=[0, kw_df_plot['H statistic'].max() * 1.45])
fig_kw.show()

# ── Chart 2: Spearman rho ──────────────────────────────────────────────────
fig_sp = px.bar(
    sp_df_plot, y='Variable', x='Spearman rho',
    text='p_fmt',
    color='Spearman rho', color_continuous_scale='RdBu_r', color_continuous_midpoint=0,
    orientation='h',
    title='Spearman ρ — Continuous Variables vs Normalized Move<br>'
          '<sup>ρ > 0: larger values → bigger normalized moves. ρ < 0: larger values → smaller moves. n shown per variable.</sup>',
    labels={'Spearman rho': 'ρ (range −1 to +1)'},
    template=TEMPLATE, height=300,
)
fig_sp.update_traces(textposition='outside', textfont_size=11)
fig_sp.update_layout(coloraxis_showscale=False,
                     xaxis_range=[sp_df_plot['Spearman rho'].min() - 0.05,
                                  sp_df_plot['Spearman rho'].max() + 0.12])
fig_sp.show()

# ── Group medians — Kruskal-Wallis ─────────────────────────────────────────
print('=' * 65)
print('GROUP MEDIANS OF normalized_move (x ATR) — Kruskal-Wallis variables')
print('=' * 65)
for row in sorted(kw_results, key=lambda r: -r['H statistic']):
    sig_label = 'SIGNIFICANT (p<0.05)' if row['Significant'] else 'not significant'
    print(f"\n{row['Variable']}  [H={row['H statistic']}, p={row['p_fmt']}]  {sig_label}")
    sorted_groups = sorted(row['grouped'].items(), key=lambda x: -float(np.median(x[1])))
    for group_name, vals in sorted_groups:
        med = float(np.median(vals))
        bar = '#' * int(med * 4)
        print(f"  {str(group_name):28s}  median={med:5.2f}x  {bar}")

print(f'\n{"─"*65}')
print('SPEARMAN rho RESULTS')
print(f'{"─"*65}')
for row in sorted(sp_results, key=lambda r: -abs(r['Spearman rho'])):
    sig_label = 'SIGNIFICANT' if row['Significant'] else 'not significant'
    direction = 'higher X -> higher normalized_move' if row['Spearman rho'] > 0 else 'higher X -> lower normalized_move'
    print(f"  {row['Variable']:28s} rho={row['Spearman rho']:+.3f}  p={row['p_fmt']}  "
          f"n={row['n']}  {sig_label}  ({direction})")

## 7 · Distributions Separated by Key Variables

### 7a · By Event Type (Gainer vs Loser)

In [ ]:
fig = make_subplots(rows=1, cols=3,
    subplot_titles=['ATR % by event type', 'Event move |%|', 'Normalized move (×ATR)'])

for etype, color in [('Gainer', COLORS['Gainer']), ('Loser', COLORS['Loser'])]:
    sub = df[df['event_type'] == etype]
    kw = dict(name=etype, marker_color=color, opacity=0.75, boxpoints='outliers')
    fig.add_trace(go.Box(y=sub['atr_pct'].clip(upper=30), **kw, showlegend=True), row=1, col=1)
    fig.add_trace(go.Box(y=sub['abs_move_pct'].clip(upper=150), **kw, showlegend=False), row=1, col=2)
    fig.add_trace(go.Box(y=sub['normalized_move'].clip(upper=20), **kw, showlegend=False), row=1, col=3)

fig.update_layout(template=TEMPLATE, height=420, boxmode='group',
                  legend=dict(orientation='h', y=1.1),
                  title_text='Gainer vs Loser — ATR, Move, and Normalized Move')
fig.show()

for col in ['atr_pct', 'abs_move_pct', 'normalized_move']:
    g = df[df['event_type']=='Gainer'][col].dropna()
    l = df[df['event_type']=='Loser'][col].dropna()
    stat, pval = stats.mannwhitneyu(g, l, alternative='two-sided')
    print(f'{col:25s}  Gainer median={g.median():.2f}  Loser median={l.median():.2f}  MW p={pval:.3e}')

### 7b · By Clinical Trial Phase — Gaussian KDE Density Curves

Each curve below is a **Gaussian kernel density estimate (KDE)** — a smoothed histogram that shows the full shape of the normalized_move distribution for that phase, all plotted on the **same x-axis** for direct comparison.

> **How to read it**: Higher Y at a given X means more events from that phase have that normalized_move.  
> The area under each curve integrates to 1 (probability density).  
> **Diamonds** mark each phase's median. **Dotted vertical lines** mark ATR class boundaries (1.5× / 3.0× / 8.0×).
>
> Compared to a box plot, the KDE shows whether the distribution has a sharp peak (most events are similar), a flat spread, or a heavy right tail (a few extreme events pulling the mean up).

In [ ]:
from scipy.stats import gaussian_kde

phase_order = ['Phase 1', 'Phase 1/Phase 2', 'Phase 2', 'Phase 2/Phase 3',
               'Phase 3', 'Phase 4', 'Unknown']
phase_order = [p for p in phase_order if p in df['phase_clean'].unique()]

# Build per-phase data (no clip for KDE computation — limit display via xaxis_range)
phase_data = {}
for p in phase_order:
    vals = df[df['phase_clean'] == p]['normalized_move'].dropna().values
    if len(vals) >= 15:
        phase_data[p] = vals

palette = px.colors.qualitative.T10[:len(phase_data)]
x_range = np.linspace(0, 20, 500)

fig_kde = go.Figure()
for (phase, vals), color in zip(phase_data.items(), palette):
    kde_fn = gaussian_kde(vals, bw_method='scott')
    density = kde_fn(x_range)
    median_val = float(np.median(vals))
    n = len(vals)

    # KDE curve
    fig_kde.add_trace(go.Scatter(
        x=x_range, y=density,
        mode='lines',
        name=f'{phase}  (n={n}, med={median_val:.1f}x)',
        line=dict(color=color, width=2.5),
        hovertemplate=f'<b>{phase}</b><br>normalized_move: %{{x:.2f}}x ATR'
                      f'<br>density: %{{y:.4f}}<extra></extra>'
    ))

    # Diamond at the median on the KDE curve
    fig_kde.add_trace(go.Scatter(
        x=[median_val],
        y=[float(kde_fn([median_val])[0])],
        mode='markers',
        marker=dict(color=color, size=10, symbol='diamond',
                    line=dict(color='white', width=1.5)),
        showlegend=False,
        hovertemplate=f'<b>{phase}</b> median: {median_val:.2f}x ATR<extra></extra>'
    ))

# ATR class boundary lines
for x_val, label in [(1.5, '1.5x  (Noise|Low)'),
                     (3.0, '3.0x  (Low|Medium)'),
                     (8.0, '8.0x  (High|Extreme)')]:
    fig_kde.add_vline(
        x=x_val, line_dash='dot',
        line_color='rgba(100,100,100,0.5)',
        annotation_text=label,
        annotation_position='top right',
        annotation_font_size=9,
    )

fig_kde.update_layout(
    title='Normalized Move (x ATR) by Trial Phase — Gaussian KDE Density Curves<br>'
          '<sup>All phases on the same axis. Diamonds = median per phase. '
          'Dotted lines = ATR class boundaries. Higher Y = more events at that x value.</sup>',
    xaxis=dict(title='Normalized move (x ATR)', range=[0, 15]),
    yaxis=dict(title='Probability density'),
    legend=dict(orientation='v', x=1.01, y=1, xanchor='left', font_size=11),
    template=TEMPLATE, height=520,
)
fig_kde.show()

# Summary statistics table
print('Normalized move by phase (sorted by median):')
phase_stats = (
    df.groupby('phase_clean')['normalized_move']
    .agg(['median', 'mean', 'count'])
    .round(2)
    .rename(columns={'median': 'median xATR', 'mean': 'mean xATR', 'count': 'n'})
)
print(phase_stats[phase_stats['n'] >= 15].sort_values('median xATR', ascending=False).to_string())

### 7c · By Market Cap Tier

In [ ]:
cap_order = ['Micro (<$100M)', 'Small ($100–500M)', 'Mid ($500M–2B)', 'Large (>$2B)']

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['ATR % by market cap (smaller = more volatile)',
                    'Normalized move by market cap (relative surprise)'])

colors_cap = ['#d63031', '#e17055', '#fdcb6e', '#00b894']
for bucket, color in zip(cap_order, colors_cap):
    sub = df[df['market_cap_bucket'] == bucket]
    kw = dict(name=bucket, marker_color=color, boxpoints='outliers', opacity=0.8)
    fig.add_trace(go.Box(y=sub['atr_pct'].clip(upper=40), **kw, showlegend=True), row=1, col=1)
    fig.add_trace(go.Box(y=sub['normalized_move'].clip(upper=15), **kw, showlegend=False), row=1, col=2)

fig.update_layout(template=TEMPLATE, height=450, boxmode='group',
                  legend=dict(orientation='h', y=-0.2),
                  title_text='Market Cap Tier — Volatility vs Normalized Move')
fig.show()

print('Market cap tier stats (normalized_move):')
for bucket in cap_order:
    sub = df[df['market_cap_bucket'] == bucket]['normalized_move'].dropna()
    if len(sub) > 0:
        print(f'  {bucket:22s}: n={len(sub):4d} | median={sub.median():.2f}× | mean={sub.mean():.2f}×')

### 7d · NCT ID Presence — Does Having a Trial ID Matter?

In [ ]:
fig = px.histogram(
    df, x='normalized_move', color='has_nct',
    barmode='overlay', nbins=60,
    color_discrete_map={True: '#0984e3', False: '#e17055'},
    labels={'normalized_move': 'Normalized move (× ATR)', 'has_nct': 'Has NCT ID'},
    title='Normalized Move: Events With vs Without NCT ID',
    template=TEMPLATE, height=400,
    range_x=[0, 20], opacity=0.7
)
fig.update_layout(legend=dict(orientation='h', y=1.1))
fig.show()

with_nct = df[df['has_nct']]['normalized_move'].dropna()
without_nct = df[~df['has_nct']]['normalized_move'].dropna()
stat, pval = stats.mannwhitneyu(with_nct, without_nct, alternative='two-sided')
print(f'With NCT:    n={len(with_nct):4d} | median={with_nct.median():.2f}× | mean={with_nct.mean():.2f}×')
print(f'Without NCT: n={len(without_nct):4d} | median={without_nct.median():.2f}× | mean={without_nct.mean():.2f}×')
print(f'Mann-Whitney p = {pval:.3e}')

## 8 · Short Interest Analysis

**`short_percent`** = % of a stock's float that is currently sold short (source: yfinance snapshot).

> **Note**: yfinance provides the **current** short interest, not the historical figure at the event date.  
> Treat as a proxy for "how controversial or disputed this stock is" rather than a precise historical value.  
> For stocks that delisted or changed drastically since the event, the signal is weaker.

**Two mechanisms by which high short interest amplifies catalyst moves:**

1. **Short squeeze (Gainers)**: A positive readout forces short sellers to buy back quickly to cover their position — adding mechanical buying pressure on top of organic demand, amplifying the upward move far beyond what fundamentals alone would suggest.

2. **Controversy / dispersion signal**: High short interest on a binary catalyst event (Phase 3 topline, FDA decision) indicates that sophisticated investors are actively betting against the drug. This market disagreement is associated with higher volatility regardless of direction — the market is divided, so the resolution of uncertainty moves it more.

**Where the data comes from**: `short_percent` was fetched via `yfinance` in `clients/financial_client.py` during enrichment. Fill rate is visible in Section 1.

In [ ]:
from scipy.stats import spearmanr as _sp

si_col = 'short_percent'
si_valid = df[df[si_col].notna()].copy()
print(f'Short interest coverage: {len(si_valid)}/{len(df)} rows '
      f'({100*len(si_valid)/len(df):.1f}%)')
print(f'Short %: median={si_valid[si_col].median():.1f}%  '
      f'mean={si_valid[si_col].mean():.1f}%  max={si_valid[si_col].max():.1f}%')

# Quartile bins
si_valid['si_quartile'] = pd.qcut(
    si_valid[si_col], q=4,
    labels=['Q1 (lowest SI)', 'Q2', 'Q3', 'Q4 (highest SI)'],
    duplicates='drop'
)

# ── Figure 1: distribution + quartile boxes ───────────────────────────────
fig1 = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Short Interest % — Distribution',
        'Normalized Move by Short Interest Quartile',
    ]
)

fig1.add_trace(go.Histogram(
    x=si_valid[si_col].clip(upper=60), nbinsx=50,
    marker_color='#6c5ce7', opacity=0.85, name='Short %',
    hovertemplate='Short %: %{x:.1f}<br>Count: %{y}'
), row=1, col=1)

colors_si = ['#74b9ff', '#fdcb6e', '#e17055', '#d63031']
for q, color in zip(['Q1 (lowest SI)', 'Q2', 'Q3', 'Q4 (highest SI)'], colors_si):
    sub = si_valid[si_valid['si_quartile'] == q]['normalized_move'].dropna()
    if len(sub) == 0:
        continue
    si_range = si_valid[si_valid['si_quartile'] == q][si_col]
    label = f'{q}<br>{si_range.min():.1f}–{si_range.max():.1f}%'
    fig1.add_trace(go.Box(
        y=sub.clip(upper=15), name=q, marker_color=color,
        boxpoints='outliers', jitter=0.3, opacity=0.85
    ), row=1, col=2)

fig1.update_layout(
    template=TEMPLATE, height=450, showlegend=False,
    title_text='Short Interest — Distribution and Effect on Normalized Move<br>'
               '<sup>Q1 = lowest short interest, Q4 = highest</sup>',
)
fig1.update_xaxes(title_text='Short interest %', row=1, col=1)
fig1.update_yaxes(title_text='Normalized move (x ATR)', row=1, col=2)
fig1.show()

# ── Figure 2: scatter short_percent vs normalized_move ────────────────────
si_nm = si_valid[['short_percent', 'normalized_move', 'event_type',
                   'ticker', 'event_date', 'ct_phase', 'abs_move_pct']].dropna(
    subset=['normalized_move'])

fig2 = px.scatter(
    si_nm,
    x='short_percent', y='normalized_move',
    color='event_type',
    color_discrete_map=COLORS,
    opacity=0.5,
    hover_data=['ticker', 'event_date', 'ct_phase', 'abs_move_pct'],
    labels={
        'short_percent': 'Short interest %',
        'normalized_move': 'Normalized move (x ATR)',
        'event_type': '',
    },
    title='Short Interest % vs Normalized Move — Gainer vs Loser<br>'
          '<sup>Higher short interest tends to correlate with larger catalyst moves '
          '(short squeeze + market controversy signal)</sup>',
    template=TEMPLATE, height=430,
)
fig2.update_layout(yaxis_range=[0, 20], legend=dict(orientation='h', y=1.1))
fig2.show()

# ── Statistics ────────────────────────────────────────────────────────────
r, pval = _sp(si_nm['short_percent'], si_nm['normalized_move'])
print(f'\nSpearman rho (short_percent vs normalized_move): rho={r:.3f}, p={pval:.2e}')
print(f'  {"SIGNIFICANT" if pval < 0.05 else "Not significant"} at p < 0.05')

print('\nNormalized move by short interest quartile:')
for q in ['Q1 (lowest SI)', 'Q2', 'Q3', 'Q4 (highest SI)']:
    sub_q = si_valid[si_valid['si_quartile'] == q]
    nm_q = sub_q['normalized_move'].dropna()
    si_q = sub_q[si_col]
    if len(nm_q) == 0:
        continue
    print(f'  {q:20s}: SI {si_q.min():.1f}–{si_q.max():.1f}%  '
          f'n={len(nm_q):4d}  median={nm_q.median():.2f}x  mean={nm_q.mean():.2f}x')

## 9 · Interactive Distribution Explorer
Use the dropdowns to split any of the three core distributions by a grouping variable.

In [ ]:
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

# Grouping variable options
group_options = {
    'event_type (Gainer/Loser)': 'event_type',
    'ML move class (Low/Med/High)': 'stock_relative_move',
    'ATR class': 'move_class_norm',
    'Trial phase': 'phase_clean',
    'Market cap tier': 'market_cap_bucket',
    'Has NCT ID': 'has_nct',
}

metric_options = {
    'Normalized move (× ATR)': ('normalized_move', 0, 20),
    'Event move |%|': ('abs_move_pct', 0, 150),
    'ATR % (baseline volatility)': ('atr_pct', 0, 35),
}

group_dd = widgets.Dropdown(
    options=list(group_options.keys()),
    value='event_type (Gainer/Loser)',
    description='Split by:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='280px')
)
metric_dd = widgets.Dropdown(
    options=list(metric_options.keys()),
    value='Normalized move (× ATR)',
    description='Metric:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='280px')
)
chart_dd = widgets.Dropdown(
    options=['Histogram (overlay)', 'Box plot', 'Violin plot'],
    value='Histogram (overlay)',
    description='Chart type:',
    style={'description_width': '80px'},
    layout=widgets.Layout(width='280px')
)
out = widgets.Output()

def update(change=None):
    col = group_options[group_dd.value]
    metric_label = metric_dd.value
    metric_col, xmin, xmax = metric_options[metric_label]
    chart_type = chart_dd.value

    plot_df = df[[col, metric_col]].dropna()
    plot_df[metric_col] = plot_df[metric_col].clip(upper=xmax)
    groups = sorted(plot_df[col].astype(str).unique())

    with out:
        out.clear_output(wait=True)
        fig = go.Figure()

        for i, grp in enumerate(groups):
            sub = plot_df[plot_df[col].astype(str) == grp][metric_col]
            color = px.colors.qualitative.Plotly[i % 10]

            if chart_type == 'Histogram (overlay)':
                fig.add_trace(go.Histogram(
                    x=sub, name=str(grp), opacity=0.7, nbinsx=50,
                    marker_color=color
                ))
                fig.update_layout(barmode='overlay')
            elif chart_type == 'Box plot':
                fig.add_trace(go.Box(
                    y=sub, name=str(grp), marker_color=color,
                    boxpoints='outliers', jitter=0.3
                ))
            else:  # Violin
                fig.add_trace(go.Violin(
                    y=sub, name=str(grp), fillcolor=color,
                    box_visible=True, meanline_visible=True,
                    opacity=0.75, line_color='white'
                ))

        axis_kw = dict(title=metric_label)
        if chart_type == 'Histogram (overlay)':
            fig.update_layout(xaxis=axis_kw)
        else:
            fig.update_layout(yaxis=axis_kw)

        fig.update_layout(
            title=f'{metric_label} — split by {group_dd.value}',
            template=TEMPLATE, height=480,
            legend=dict(orientation='h', y=1.1)
        )
        fig.show()

        # Stats table
        rows_stats = []
        for grp in groups:
            sub = plot_df[plot_df[col].astype(str) == grp][metric_col]
            rows_stats.append({'Group': grp, 'n': len(sub),
                               'median': round(sub.median(), 2),
                               'mean': round(sub.mean(), 2),
                               'p25': round(sub.quantile(0.25), 2),
                               'p75': round(sub.quantile(0.75), 2),
                               'p90': round(sub.quantile(0.90), 2)})
        display(pd.DataFrame(rows_stats))

group_dd.observe(update, names='value')
metric_dd.observe(update, names='value')
chart_dd.observe(update, names='value')

display(widgets.HBox([group_dd, metric_dd, chart_dd]))
display(out)
update()

## 10 · Key Takeaways

| Finding | Detail |
|---|---|
| **Most CT.gov completions have minimal impact** | Median normalized_move ≈ 0.6× ATR — the stock barely moved |
| **The tail is fat** | p95 ≈ 7× ATR, max ≈ 39× ATR — extreme outliers dominate the mean |
| **ATR% strongly predicts absolute move** | But normalized move equalizes across stocks — more comparable |
| **Phase drives move size** | Phase 3 events produce larger normalized moves than Phase 1/2 |
| **Small caps have higher ATR but similar normalized moves** | Relative surprise is roughly equal — ATR normalization works |
| **NCT presence signal** | Events without NCT IDs (Perplexity batch) tend to be higher movers — they were specifically sourced as large moves |
| **Short interest amplifies moves** | Higher short % correlates with larger normalized moves (squeeze + controversy signal) |

---
*Dataset: `enriched_all_clinical.csv` — 2,175 clinical catalyst events | Biotech Catalyst Pipeline v3.4*

---
## 📖 Column Reference & Findings Documentation

### Core Movement Columns

| Column | Type | Fill % | Description |
|--------|------|--------|-------------|
| `move_pct` | float | 98.8% | Stock % change on the catalyst event day. Positive = Gainer, negative = Loser. Raw value, not absolute. |
| `abs_move_pct` | float | 98.8% | `abs(move_pct)` — absolute size of the move regardless of direction. Used in Section 5 scatter Y axis. |
| `move_2d_pct` | float | 75.5% | 2-day cumulative % move after the event (captures late reactions / overnight gaps). |
| `price_before` | float | 75.5% | Stock price the day before the event. |
| `price_after` | float | 75.5% | Stock price the day after the event. |

---

### ATR (Average True Range) Columns

> **What is ATR?** Average True Range measures a stock's *typical* daily price range, incorporating gap opens. It is the standard volatility baseline in technical analysis. We use **Wilder's ATR** (exponential moving average with α = 1/20), computed over the **20 trading days strictly before the event** — no look-ahead bias.
>
> ATR ≈ 0.8σ of daily log returns. So 1× ATR ≈ 1.25σ, meaning moves above 3× ATR occur by chance in <0.02% of normal trading days.

| Column | Type | Fill % | Description |
|--------|------|--------|-------------|
| `atr_pct` | float | 98.5% | ATR expressed as % of the stock's closing price. Represents *how much this stock typically moves on any given day*. A stock with `atr_pct = 5.0` moves ~5% on a normal day. Large-cap pharma typically 1–3%; small-cap biotech 5–20%+. |
| `atr_value` | float | 0% | ATR in raw dollar terms. Not yet filled — requires `recompute_atr.py` run with yfinance access. |
| `avg_daily_move` | float | 98.5% | Mean of `abs(close-to-close %)` over the same 20-day window. Simpler than ATR — doesn't use High/Low intraday range. Usually slightly lower than `atr_pct`. |

---

### ATR-Normalized Move Columns

> **Core insight**: A 20% move on a stock with 2% ATR is 10× its normal range — extraordinary. The same 20% on a stock with 15% ATR is 1.3× — barely unusual. Raw % comparisons across different stocks are misleading; ATR-normalized comparisons are not.

| Column | Type | Fill % | Description |
|--------|------|--------|-------------|
| `normalized_move` | float | 98.5% | `abs(move_pct) / atr_pct` — **how many ATRs the event move represents**. This is the X-fold multiple. Example: `normalized_move = 6.5` means the stock moved 6.5× its typical daily range. |
| `move_class_norm` | string | 98.5% | ATR-multiple bucket (5 levels). See table below. |
| `move_class_abs` | string | 98.5% | Absolute % bucket (5 levels). Does NOT account for the stock's own volatility. |
| `move_class_combo` | string | 98.5% | Combined ML label = same as `stock_relative_move`. Uses dual AND rule. |
| `stock_relative_move` | string | 98.8% | **Primary ML label.** Alias for `move_class_combo`. Categories: `Low`, `Medium`, `High`. |
| `move_magnitude` | string | — | Legacy column, superseded by `move_class_norm`. Ignore. |

#### `move_class_norm` — ATR Multiple Buckets

| Value | Threshold | Approx σ equivalent | Frequency by chance |
|-------|-----------|---------------------|---------------------|
| `Noise` | < 1.5× ATR | < 1.9σ | ~6% of normal days |
| `Low` | 1.5–3× ATR | 1.9–3.75σ | ~0.02–5% |
| `Medium` | 3–5× ATR | 3.75–6.25σ | Very rare |
| `High` | 5–8× ATR | 6.25–10σ | Extremely rare |
| `Extreme` | ≥ 8× ATR | ≥ 10σ | Essentially impossible by chance |

#### `stock_relative_move` / `move_class_combo` — ML Label Logic

```
Low    = abs(move_pct) < 15%  AND  normalized_move < 2.5×   → routine, no meaningful signal
High   = abs(move_pct) ≥ 20%  AND  normalized_move ≥ 3.5×   → both large AND statistically unusual
Medium = everything else                                      → ambiguous / mixed signal
```

The **dual AND requirement** prevents two failure modes:
- A highly volatile stock (ATR = 20%) with a 25% move = 1.25× ATR → would be called High by pure %, but is barely unusual for that stock → correctly labelled **Medium**
- A stable stock (ATR = 1%) with a 10% move = 10× ATR → purely ATR-based would call this High, but it's below our 20% minimum → correctly labelled **Medium**

---

### Key Statistical Findings from This Dataset

| Finding | Evidence |
|---------|----------|
| **Most CT.gov completion events have minimal stock impact** | Median `normalized_move` = 0.61× ATR — the stock barely moved beyond its daily noise level |
| **The distribution is extremely right-skewed** | Mean = 1.98× but median = 0.61×; p90 = 5.9×, max = 39× |
| **Phase 3 events produce systematically larger normalized moves** | Kruskal-Wallis confirms phase is a significant predictor (p < 0.05) |
| **Small-cap stocks have higher `atr_pct` but similar `normalized_move`** | ATR normalization successfully equalizes across market cap tiers — it's a fairer comparison |
| **Events without NCT IDs are higher movers** | These come from the Perplexity/news scan, which specifically targeted large moves — selection bias |
| **Short interest has a small but significant positive correlation** | Higher short % → larger normalized moves (squeeze amplification + catalyst signal combine) |
| **`stock_relative_move = High` accounts for only ~18% of events** | But these represent the tail that matters most for trading signal research |

---
*Pipeline: `biotech_catalyst_v3` | ATR implementation: `utils/volatility.py` | Data: `enriched_all_clinical.csv`*